In [ ]:
# CALCULATES CIRCULATIONS AND VALIDATIONS PER MUNICIPALITY (per day in a month) BASED ON OPERATION PLAN OF THAT MONTH


# # create conection 
# get offer plan of a month
#get municipalities shapes
#count total circulations per stop and day
#get table municipality-day-num_circ
#import validations data of that month (calculated in notebook get_validations_by_agency_stop_day.ipynb)
#merge circulations and validations per day and municipality
#merge circulations and validations per day and agency




### Set Global variables

In [ ]:
# =========================
# FILTERS FOR DATA EXTRACTION
# =========================
AGENCY_IDS = {"41", "42", "43", "44"}

START_DATE = 20260101
END_DATE   = 20260228

# =========================
# GLOBAL VARS TO GET FROM CONFIG FILE
# =========================

required_vars = [

    "TIMEZONE",
    "OPERATION_PLAN_INPUT_FILES_FOLDER"
    "OUTPUT_FILES_FOLDER",
]

In [ ]:
# from config py file

import importlib
import config

# import, cleaning the cache to get latest changes
importlib.reload(config)


print("Config file in:", config.__file__)


missing = []
locals_dict = locals()

for name in required_vars:
    if hasattr(config, name):
        locals_dict[name] = getattr(config, name)
    else:
        missing.append(name)

if missing:
    raise RuntimeError(f"Missing config variables: {missing}")

# Create directory for output files if it doesn't exist
import os
from pathlib import Path
OUTPUT_FILES_FOLDER.mkdir(parents=True, exist_ok=True)

[name for name in required_vars if hasattr(config, name) and print(name, getattr(config, name))]

In [ ]:


# folder to save operation plans input files
import os
from pathlib import Path


import shutil

# set operation files folder path
# if folder exists, delete it and create a new one
if OPERATION_PLAN_INPUT_FILES_FOLDER.exists():
    shutil.rmtree(OPERATION_PLAN_INPUT_FILES_FOLDER)
OPERATION_PLAN_INPUT_FILES_FOLDER.mkdir(parents=True, exist_ok=True)


# source of operation plans
API_URL = "https://go.tmlmobilidade.pt/plans/api/plans/approved"

# source of municipalities shapefile
MUNICIPALITIES_SHP_URL = "https://github.com/carrismetropolitana/datasets/blob/latest/locations/municipalities.zip?raw=true"



### Get operation plans

In [ ]:
#Fetch API data
import requests

response = requests.get(API_URL, timeout=30)
response.raise_for_status()

plans = response.json().get("data", [])
print(f"Total plans found: {len(plans)}")


In [ ]:
#list all available plans
# Fetch API data
import requests
from pathlib import Path

response = requests.get(API_URL, timeout=30)
response.raise_for_status()

plans = response.json().get("data", [])
print(f"Total plans found: {len(plans)}")

results = []

for plan in plans:
    agency_id = plan.get("gtfs_agency", {}).get("agency_id")
    feed_info = plan.get("gtfs_feed_info", {})
    file_url  = plan.get("operation_file_url")

    if not agency_id or not feed_info or not file_url:
        continue

    try:
        feed_start = int(feed_info.get("feed_start_date"))
        feed_end   = int(feed_info.get("feed_end_date"))
    except (TypeError, ValueError):
        continue

    filename = Path(file_url).name

    results.append({
        "filename": filename,
        "agency_id": agency_id,
        "feed_start_date": feed_start,
        "feed_end_date": feed_end,
    })

# Order by feed_start_date
results.sort(key=lambda x: x["feed_start_date"])

# Output message
print("\nPlans ordered by feed_start_date:")
for r in results:
    print(
        f"{r['filename']} | "
        f"agency_id={r['agency_id']} | "
        f"feed_start_date={r['feed_start_date']} | "
        f"feed_end_date={r['feed_end_date']}"
    )


In [ ]:
# Filter plans and download ZIP files

#END_DATE included 


downloaded = []

for plan in plans:
    agency_id = plan.get("gtfs_agency", {}).get("agency_id")
    feed_info = plan.get("gtfs_feed_info", {})
    file_url  = plan.get("operation_file_url")

    if not agency_id or not feed_info or not file_url:
        continue

    if agency_id not in AGENCY_IDS:
        continue

    try:
        feed_start = int(feed_info.get("feed_start_date"))
        feed_end   = int(feed_info.get("feed_end_date"))
    except (TypeError, ValueError):
        continue

    if feed_start <= END_DATE and feed_end >= START_DATE:
        filename = Path(file_url).name
        output_path = OPERATION_PLAN_INPUT_FILES_FOLDER / filename

        meta_msg = (
            f"agency_id={agency_id}, "
            f"feed_start_date={feed_start}, "
            f"feed_end_date={feed_end}"
        )

        if output_path.exists():
            print(f"Already exists: {filename} ({meta_msg})")
            continue

        print(f"Downloading: {filename} ({meta_msg})")

        with requests.get(file_url, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        downloaded.append({
            "filename": filename,
            "agency_id": agency_id,
            "feed_start_date": feed_start,
            "feed_end_date": feed_end,
        })


In [ ]:
print("\nDownload completed")
print(f"Files downloaded: {len(downloaded)}")

for f in downloaded:
    print(" -", f)


### Get data from operation plan

In [ ]:
import pandas as pd
import zipfile
from pathlib import Path

# ---- Utility to read specific columns from a CSV inside a ZIP ----
def read_csv_from_zip(zip_file: zipfile.ZipFile, filename: str, usecols: list):
    with zip_file.open(filename) as f:
        return pd.read_csv(f, usecols=usecols)

# ---- Process one GTFS ZIP ----
def extract_trip_stop_dates_from_gtfs_zip(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as z:
        files = set(z.namelist())
        required_files = {"trips.txt", "routes.txt", "calendar_dates.txt", "stop_times.txt"}

        if not required_files <= files:
            print(f"Skipping {zip_path.name}, missing files: {required_files - files}")
            return pd.DataFrame()

        # --- Load routes (for agency_id) ---
        routes = read_csv_from_zip(z, "routes.txt", ["route_id", "agency_id"])

        # --- Load trips ---
        trips = read_csv_from_zip(z, "trips.txt", ["trip_id", "route_id", "service_id"])
        trips = trips.merge(routes, on="route_id", how="left")

        # --- Load calendar_dates (only exception_type == 1) ---
        cal_dates = read_csv_from_zip(z, "calendar_dates.txt", ["service_id", "date", "exception_type"])
        cal_dates = cal_dates[cal_dates["exception_type"] == 1][["service_id", "date"]]

        if cal_dates.empty:
            return pd.DataFrame()  # No trips to keep

        # --- Load stop_times ---
        stop_times = read_csv_from_zip(z, "stop_times.txt", ["trip_id", "stop_id"])

        # --- Merge: trips ↔ stops ↔ calendar_dates ---
        df = (
            trips.merge(stop_times, on="trip_id", how="inner")
                 .merge(cal_dates, on="service_id", how="inner")
        )

        # --- Keep only needed columns and remove duplicates ---
        df = df[["agency_id", "trip_id", "stop_id", "date"]].drop_duplicates()
        return df

# ---- Incrementally process all ZIP files ----
all_results = []

zip_files = sorted(OPERATION_PLAN_INPUT_FILES_FOLDER.glob("*.zip"))
print(f"GTFS ZIP files found: {len(zip_files)}")

for zip_path in zip_files:
    print(f"Processing: {zip_path.name}")
    df = extract_trip_stop_dates_from_gtfs_zip(zip_path)
    if not df.empty:
        all_results.append(df)

# ---- Combine all results into a unique dataframe ----
if all_results:
    trip_stop_date = pd.concat(all_results, ignore_index=True)
    trip_stop_date.drop_duplicates(inplace=True)
    print(f"Final dataframe shape: {trip_stop_date.shape}")
else:
    trip_stop_date = pd.DataFrame(columns=["agency_id", "trip_id", "stop_id", "date"])
    print("No data extracted from ZIPs.")

# ---- Optional: quick preview ----
print(trip_stop_date.head(3))


#### Get stops from operation plan

In [ ]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from zipfile import ZipFile

folder_path = OPERATION_PLAN_INPUT_FILES_FOLDER

# Generator to yield dataframes from each GTFS zip
def iter_gtfs_stops(folder):
    for file_name in os.listdir(folder):
        if file_name.endswith('.zip'):
            zip_path = os.path.join(folder, file_name)
            with ZipFile(zip_path) as zf:
                if 'stops.txt' in zf.namelist():
                    # Read only necessary columns
                    df = pd.read_csv(zf.open('stops.txt'), usecols=['stop_id', 'stop_lat', 'stop_lon'])
                    yield df

# Concatenate all stops in chunks to reduce memory usage
dfs = iter_gtfs_stops(folder_path)
all_stops_df = pd.concat(dfs, ignore_index=True)

# Keep only distinct stop_ids
all_stops_df = all_stops_df.drop_duplicates(subset=['stop_id'])

# Convert to GeoDataFrame
all_stops_gdf = gpd.GeoDataFrame(
    all_stops_df,
    geometry=gpd.points_from_xy(all_stops_df.stop_lon, all_stops_df.stop_lat),
    crs="EPSG:4326"
)

print(all_stops_gdf.head())
print(f"Total distinct stops: {len(all_stops_gdf)}")


### Get municipalities

#### Get municipalities shapes

In [ ]:
from pathlib import Path
import requests


output_municipalities_shp_zip = OUTPUT_FILES_FOLDER / "municipalities.zip"

with requests.get(MUNICIPALITIES_SHP_URL, stream=True) as r:
    r.raise_for_status()
    with open(output_municipalities_shp_zip, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print(f"Downloaded to {output_municipalities_shp_zip}")

In [ ]:
import geopandas as gpd
import zipfile
import os

zip_path = os.path.join(OUTPUT_FILES_FOLDER, "municipalities.zip")

# Find the JSON file inside the zip
with zipfile.ZipFile(zip_path) as z:
    json_files = [f for f in z.namelist() if f.endswith(".json")]

if not json_files:
    raise ValueError("No .json file found in municipalities.zip")

# Read the first JSON (or loop if there are many)
json_in_zip = json_files[0]

gdf_municipalities = gpd.read_file(f"zip://{zip_path}!{json_in_zip}")

print(gdf_municipalities.head())
print(gdf_municipalities.crs)


#### Get municipality for stops

In [ ]:
# Reproject municipalities if needed
if all_stops_gdf.crs != gdf_municipalities.crs:
    old_crs = gdf_municipalities.crs
    gdf_municipalities = gdf_municipalities.to_crs(all_stops_gdf.crs)
    print(
        f"Municipalities reprojected "
        f"from {old_crs} to {all_stops_gdf.crs}"
    )
else:
    print("Municipalities CRS already matches stops CRS – no reprojection needed")


In [ ]:
import geopandas as gpd
import pandas as pd

# Only municipality columns we need
gdf_municipalities_sub = gdf_municipalities[
    ["id", "name", "geometry"]
].rename(columns={
    "id": "municipality_id",
    "name": "municipality_name"
})

# Build spatial indexes explicitly (important for speed)
_ = gdf_municipalities_sub.sindex
_ = all_stops_gdf.sindex


In [ ]:
#Chunked Spatial Join (memory-safe)
def chunked_spatial_join(
    points_gdf,
    polygons_gdf,
    chunk_size=100_000,
    predicate="within"
):
    result_chunks = []

    for start in range(0, len(points_gdf), chunk_size):
        end = start + chunk_size
        points_chunk = points_gdf.iloc[start:end]

        joined = gpd.sjoin(
            points_chunk,
            polygons_gdf,
            how="left",
            predicate=predicate
        )

        # Drop join helper column
        joined = joined.drop(columns="index_right")

        result_chunks.append(joined)

    return gpd.GeoDataFrame(
        pd.concat(result_chunks, ignore_index=True),
        crs=points_gdf.crs
    )


In [ ]:
# join stops - municipalities
stops_municip_gdf = chunked_spatial_join(
    points_gdf=all_stops_gdf,
    polygons_gdf=gdf_municipalities_sub,
    chunk_size=100_000,     # tune based on RAM
    predicate="within"     # or "intersects"
)

print(stops_municip_gdf.head())


In [ ]:
stops_municip = stops_municip_gdf[
                ["stop_id", "municipality_id", "municipality_name"]].drop_duplicates()
stops_municip

In [ ]:

stops_municip["municipality"] = (
    stops_municip["municipality_name"]
    + "_" +
    stops_municip["municipality_id"].astype(str)
)

In [ ]:
stops_municip.head(3)   

### Get circulations per day

In [ ]:
trip_stop_date.head(3)

#### Count circulations by day by municipality

### Import validations data (from notebook get_validations_by_agency_stop_day)

In [ ]:
# import validations data

# validations data structure
# date,agency_id,stop_id,validations

validations = pd.read_csv(os.path.join(OUTPUT_FILES_FOLDER, "date_agency_stop_validations.csv"))

validations.head(3) 


### Merge circulations and validations data

In [ ]:
import pandas as pd

# --- Join trip stops with municipalities ---
trip_stop_municip = (
    trip_stop_date
    .merge(
        stops_municip[[
            "stop_id",
            "municipality_id",
            "municipality_name",
            "municipality",
        ]],
        on="stop_id",
        how="inner",
    )
)

# --- Join validations with municipalities ---
validations_municip = (
    validations
    .merge(
        stops_municip[[
            "stop_id",
            "municipality_id",
            "municipality_name",
            "municipality",
        ]],
        on="stop_id",
        how="inner",
    )
)

# ============================================================
# 1️ MUNICIPALITY per day
# ============================================================

# Circulations = distinct trips
municip_circulations = (
    trip_stop_municip[
        ["date", "municipality", "trip_id"]
    ]
    .drop_duplicates()
    .groupby(["date", "municipality"])
    .size()
    .reset_index(name="circulations")
)

# Validations = sum
municip_validations = (
    validations_municip
    .groupby(["date", "municipality"], as_index=False)
    .agg(validations=("validations", "sum"))
)

municipality_circulations_validations_perday = (
    municip_circulations
    .merge(
        municip_validations,
        on=["date", "municipality"],
        how="inner",
    )
    .fillna(0)
)

# ============================================================
# 2 AGENCY per day
# ============================================================

# Circulations = distinct trips
agency_circulations = (
    trip_stop_municip[
        ["date", "agency_id", "trip_id"]
    ]
    .drop_duplicates()
    .groupby(["date", "agency_id"])
    .size()
    .reset_index(name="circulations")
)

# Validations = sum
agency_validations = (
    validations_municip
    .groupby(["date", "agency_id"], as_index=False)
    .agg(validations=("validations", "sum"))
)

agency_circulations_validations_perday = (
    agency_circulations
    .merge(
        agency_validations,
        on=["date", "agency_id"],
        how="inner",
    )
    .fillna(0)
)



In [ ]:
municipality_circulations_validations_perday.tail(3)

In [ ]:
agency_circulations_validations_perday.head(3)

In [ ]:
#save to CSV

municipality_circulations_validations_perday.to_csv(os.path.join(OUTPUT_FILES_FOLDER, "municipality_circulations_validations_perday.csv"),
     index=False
 )


agency_circulations_validations_perday.to_csv(os.path.join(OUTPUT_FILES_FOLDER, "agency_circulations_validations_perday.csv"),
     index=False
 )